# Step 3: 部署性能压测 — vllm bench serve + /metrics 显存拆分

**目标**：掌握量化部署的性能验证方法——构造 `vllm bench serve` 压测命令、从 `/metrics` 解析 KV-Cache 占用与权重显存拆分、理解"为什么测 TTFT 不能用 `llm.generate()`"和"为什么 nvidia-smi 差值法会高估 KV-Cache"，最终用 FP16 基线对比量化的吞吐/显存红利。

**对应 OUTLINE 课时**：4.3 部署性能压测（~40 分钟）。

> **声明式主线**：压测不改模型——本节你写的是 `vllm bench serve` 命令 + metrics 解析。L3 真压测跨模块读 M2 7B 量化产物（三量化）+ FP16 基线（7B 本身）四向对比。


## 学完应能讲清（学完本节应能口头回答）

1. `vllm bench serve` / `throughput` / `latency` 三个分别测什么？为什么测 **TTFT 不能用 `llm.generate()`**（V1 引擎下不可靠）？（提示：serve 测在线服务真实负载下的 TTFT/吞吐；throughput/latency 测离线批量；`llm.generate()` 不走调度器在线路径，TTFT 失真——TTFT 必须起 server 用 serve 测）
2. 读 `/metrics` 的 `gpu_cache_usage_perc` / `num_gpu_blocks` 看 KV-Cache 占用，为什么 **nvidia-smi 差值法会高估** KV-Cache？（提示：nvidia-smi 显示进程已映射显存含预分配池，非实际 KV 用量；`/metrics` 是调度器真实账本）
3. 量化的性能红利（吞吐/显存）怎么压测验证？为什么必须有 **FP16 基线对比**？（提示：单看量化数字无意义——要和 FP16 同条件对比才知道"量化省了多少显存、提速多少/拖慢多少"；这也是为什么 s1 保留 7B FP16 本身作基线）


In [ ]:
%%capture
import pathlib, os, json, shlex
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 7B 量化产物做 L3；0.5B 兜底 L2。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"          # FP16 基线（四向对比之一）
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"         # L2/L3 轻量验证兜底
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"      # qwen7b-fp8 / qwen7b-awq / qwen7b-smoothquant
M3_OUT = REPO_COURSE / "m3-tuning-eval" / "out"
print("MODULE_ROOT =", MODULE_ROOT)
print("M2_OUT =", M2_OUT, "| exists:", M2_OUT.exists())


## 原理：性能压测三件套 + 显存拆分

**三个 bench 子命令测什么**：

| 子命令 | 测什么 | 路径 |
|---|---|---|
| `vllm bench serve` | **在线服务**真实负载下的 TTFT / 吞吐 / 延迟（起 server + 模拟客户端并发）| server + client |
| `vllm bench throughput` | **离线批量**吞吐（一次性塞一批 prompt，看 token/s）| 离线 LLM |
| `vllm bench latency` | **离线单请求**延迟 | 离线 LLM |

**为什么 TTFT 不能用 `llm.generate()`**：TTFT（Time To First Token）是"客户端发请求到收到第一个 token"的端到端延迟，必须经过 vLLM 调度器的在线路径（请求队列 + prefill 调度 + 首 token 生成）。`llm.generate()` 是离线 batch 接口，不走在线调度器，V1 引擎下其 TTFT 不可靠（调度行为不同）。**所以 TTFT 必须起 server 用 `vllm bench serve` 测**。

**显存拆分**（4.3 关键洞察）：推理显存 = 权重 + KV-Cache + 激活。量化的红利主要在**权重**（W4A16 把 7B 从 ~14GB 压到 ~5GB），但长上下文/大 batch 下 **KV-Cache** 反成大头。要看清这个拆分：

- **读 `/metrics`**（vLLM Prometheus 端点）：`vllm:gpu_cache_usage_perc`（KV-Cache 利用率）、`vllm:num_gpu_blocks`（已用 KV 块）、`vllm:max_gpu_blocks`（KV 块上限）。这是调度器的**真实账本**。
- **nvidia-smi 差值法会高估**：`nvidia-smi` 显示的是进程**已映射显存**（含 vLLM 预分配的 KV 池，即使没用满也显示已占），所以"空载记基线、峰值记高"的差值法会把预分配池算进 KV 用量，**高估** KV-Cache。`/metrics` 的 `gpu_cache_usage_perc` 才是真实分配比率。

**为什么必须有 FP16 基线**：单看量化数字无意义——"7B AWQ 占 5GB"脱离基线没法判断"省了多少"。必须**同 seed/prompt/num_prompts/TP** 下对比 FP16 vs 三量化，才能量化"省显存 X GB / 提速 Y% / 拖慢 Z%"。这就是 s1 保留 7B FP16 本身的原因。


## 亲手摸一摸：bench 子命令 + /metrics 字段名

看 `vllm bench` 的子命令、`/metrics` 真实返回的字段名——为构造压测命令和解析 metrics 打底。


In [ ]:
# 摸一摸：vllm bench 子命令 + 一段示例 /metrics 文本（看字段名）
# 一段真实形态的 /metrics 文本（教学样本，不起服务）
SAMPLE_METRICS = """\
# HELP vllm:gpu_cache_usage_perc GPU cache 的使用率
# TYPE vllm:gpu_cache_usage_perc gauge
vllm:gpu_cache_usage_perc 0.4213
# HELP vllm:num_gpu_blocks 已用 GPU KV 块数
# TYPE vllm:num_gpu_blocks gauge
vllm:num_gpu_blocks 8421
# HELP vllm:max_gpu_blocks GPU KV 块上限
vllm:max_gpu_blocks 20000
# HELP vllm:gpu_prefix_cache_queries_total 前缀缓存命中查询数
# TYPE vllm:gpu_prefix_cache_queries_total counter
vllm:gpu_prefix_cache_queries_total 150.0
vllm:gpu_prefix_cache_queries_total 30.0
# HELP vllm:num_requests_running 正在运行的请求数
vllm:num_requests_running 7
"""
print("=== /metrics 关键字段名（vLLM Prometheus 端点）===")
for field in ["vllm:gpu_cache_usage_perc", "vllm:num_gpu_blocks", "vllm:max_gpu_blocks",
              "vllm:gpu_prefix_cache_queries_total", "vllm:num_requests_running"]:
    print(" ", field)
print("\n=== 示例 metrics 文本（前 6 行）===")
print("\n".join(SAMPLE_METRICS.splitlines()[:6]))

print("\n=== vllm bench 子命令 ===")
for sub in ["serve", "throughput", "latency"]:
    print("  vllm bench", sub)
print("（serve 测在线 TTFT/吞吐；throughput/latency 测离线批量/单请求）")


## 本步填空（2 个）

1. **`build_bench_cmd(base_url, model, dataset='sharegpt', num_prompts=1000)`** — 构造 `vllm bench serve` 命令。**为什么这么设计（填前先想）**：压测是 4.3 的核心动作；学员组参数（dataset/num_prompts/base_url/model）理解压测要控什么变量——固定这些才能让 FP16 vs 量化对比可比。
2. **`parse_metrics(metrics_text)`** — 从 `/metrics` Prometheus 文本解析 KV-Cache 占用 + 权重显存拆分，返回 dict。**为什么这么设计**：显存拆分是 4.3 的关键洞察（权重 vs KV-Cache）；学员亲手解析 metrics 文本，理解"为什么不能信 nvidia-smi"。


In [ ]:
def build_bench_cmd(base_url, model, dataset="sharegpt", num_prompts=1000):
    """构造 vllm bench serve 命令字符串（空格拼接，含空格的值用 shlex.quote）。

    为什么这么设计（填前先想）：
    - 压测是 4.3 核心动作。要控的变量（让 FP16 vs 量化可比）：
        --backend openai            用 OpenAI 兼容协议连服务端
        --base-url <url>            服务地址（如 http://localhost:8000）
        --model <name>              模型名（vllm serve 启动时的 model 参数）
        --dataset <name>            压测数据集（sharegpt 模拟真实对话分布）
        --num-prompts <n>           压测请求数（1000 统计稳定）
    - 这些固定后，唯一变量才是「模型本身」（FP16 vs 三量化），对比才有意义。

    返回：命令字符串（如 'vllm bench serve --backend openai --base-url ...'）。
    """
    # TODO: parts 从 ['vllm','bench','serve'] 起，追加上面 5 组 flag（名+值成对），
    #   ' '.join，值含空格用 shlex.quote。
    raise NotImplementedError


In [ ]:
def parse_metrics(metrics_text):
    """从 /metrics Prometheus 文本解析显存拆分相关字段，返回 dict。

    为什么这么设计（填前先想）：
    - /metrics 是 vLLM 调度器的真实账本（vs nvidia-smi 的已映射显存会高估）。
      你要会读它才能讲清「权重 vs KV-Cache 谁是大头」。
    - Prometheus 文本格式：每行 'name value' 或 'name value timestamp'，# 开头是注释。
      如：
          # HELP ... 注释
          vllm:gpu_cache_usage_perc 0.4213
          vllm:num_gpu_blocks 8421 1234567890
    - 要解析的字段（取 value，忽略可选 timestamp）：
        vllm:gpu_cache_usage_perc       -> kv_cache_usage_perc (float)
        vllm:num_gpu_blocks             -> num_gpu_blocks (int)
        vllm:max_gpu_blocks             -> max_gpu_blocks (int)
        vllm:gpu_prefix_cache_queries_total -> 累加（counter 可能多行）-> prefix_cache_queries_total (float)
    - 额外推导：若同时有 num_gpu_blocks 和 max_gpu_blocks，算 kv_cache_block_util = num/max。

    返回：dict（含上述键；缺失的字段不出现）。
    """
    # TODO: 逐行解析：
    #   1) 跳过空行 / # 注释。
    #   2) split 取 name=value（value 是 parts[1]，转 float；非数值跳过）。
    #   3) 按 name 映射到上面字段；prefix_cache_queries_total 是 counter 要累加（多行相加）。
    #   4) 有 num+max 则算 kv_cache_block_util。
    raise NotImplementedError


In [ ]:
%%ipytest -qq

def test_build_bench_cmd_basic():
    cmd = build_bench_cmd("http://localhost:8000", "qwen7b-fp8")
    assert cmd.startswith("vllm bench serve")
    assert "--backend openai" in cmd
    assert "--base-url http://localhost:8000" in cmd
    assert "--model qwen7b-fp8" in cmd
    assert "--dataset sharegpt" in cmd          # 默认 dataset
    assert "--num-prompts 1000" in cmd          # 默认 num_prompts

def test_build_bench_cmd_custom():
    cmd = build_bench_cmd("http://1.2.3.4:8000", "m", dataset="random", num_prompts=500)
    assert "--base-url http://1.2.3.4:8000" in cmd
    assert "--dataset random" in cmd
    assert "--num-prompts 500" in cmd

def test_parse_metrics_usage_perc():
    r = parse_metrics(SAMPLE_METRICS)
    assert r["kv_cache_usage_perc"] == 0.4213
    assert r["num_gpu_blocks"] == 8421
    assert r["max_gpu_blocks"] == 20000

def test_parse_metrics_block_util():
    r = parse_metrics(SAMPLE_METRICS)
    # num/max = 8421/20000
    assert abs(r["kv_cache_block_util"] - 8421/20000) < 1e-9

def test_parse_metrics_prefix_cache_counter_summed():
    r = parse_metrics(SAMPLE_METRICS)
    # counter 多行 150 + 30 = 180
    assert r["prefix_cache_queries_total"] == 180.0

def test_parse_metrics_ignores_comments_and_timestamps():
    txt = "# comment line\nvllm:gpu_cache_usage_perc 0.5 1700000000\n"
    r = parse_metrics(txt)
    assert r["kv_cache_usage_perc"] == 0.5

def test_parse_metrics_missing_field_not_present():
    r = parse_metrics("vllm:num_gpu_blocks 100\n")
    assert "kv_cache_usage_perc" not in r
    assert r["num_gpu_blocks"] == 100


## L2（CPU）：命令构造 + metrics 解析（纯逻辑）

L2 验 `build_bench_cmd` 命令模板 + `parse_metrics` 解析逻辑（CPU 可跑，喂示例 metrics 文本，不起服务）。


In [ ]:
## L2：构造 bench 命令（FP16 + 三量化四向）+ 解析示例 metrics
print("=== L2：四向压测命令（FP16 基线 + 三量化）===")
four = {
    "FP16":         MODEL_DIR,
    "FP8":          M2_OUT / "qwen7b-fp8",
    "AWQ":          M2_OUT / "qwen7b-awq",
    "SmoothQuant":  M2_OUT / "qwen7b-smoothquant",
}
for name, path in four.items():
    avail = (path / "config.json").exists() if path != MODEL_DIR else MODEL_DIR.exists()
    tag = "（真 M2 产物）" if avail and path != MODEL_DIR else ("（FP16 基线）" if path == MODEL_DIR else "（缺失，先跑 M2）")
    cmd = build_bench_cmd("http://localhost:8000", str(path), num_prompts=1000)
    print("\n[%s] %s" % (name, tag))
    print("  " + cmd)

print("\n=== L2：解析示例 /metrics（显存拆分）===")
parsed = parse_metrics(SAMPLE_METRICS)
print("解析结果:", parsed)
print("  -> KV-Cache 利用率 = %.1f%%（gpu_cache_usage_perc，调度器真实账本）" % (parsed["kv_cache_usage_perc"]*100))
print("  -> KV 块利用 = %.1f%%（num/max 推导）" % (parsed["kv_cache_block_util"]*100))
print("  -> 对比：nvidia-smi 差值法会把预分配 KV 池算进去，高估真实用量。")
assert parsed["kv_cache_usage_perc"] == 0.4213
assert parsed["prefix_cache_queries_total"] == 180.0
print("\nL2 通过：bench 命令模板 + metrics 解析逻辑正确（真压测见 L3）。")


## L3（H200，GPU + SKIP_L3 双守卫）：真压测 FP16 + 三量化四向对比

L3 起服务后对 FP16 + FP8/AWQ/SmoothQuant 四向 `vllm bench serve`，输出吞吐/TTFT/显存对比表。

> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过（起 4 个服务 + 压测极重）；真人跑时不设，L3 实证。


In [ ]:
import torch, os, subprocess, time, signal, json

def run_l3_benchmark():
    four = {"FP16": MODEL_DIR, "FP8": M2_OUT / "qwen7b-fp8",
            "AWQ": M2_OUT / "qwen7b-awq", "SmoothQuant": M2_OUT / "qwen7b-smoothquant"}
    available = {n: p for n, p in four.items() if (p / "config.json").exists()}
    if len(available) < 2:
        print("[L3] 可用模型 <2（需 FP16 + 至少一个量化产物），跳过四向对比。")
        print("  先跑 M2 产出 FP8/AWQ/SmoothQuant 7B，再回 s3 跑 L3。")
        return
    results = {}
    for name, path in available.items():
        serve_cmd = build_serve_cmd_str(path, tp=1)  # 单卡压测
        print("\n[L3] %s：起服务 %s" % (name, path))
        proc = subprocess.Popen(serve_cmd, shell=True, preexec_fn=os.setsid,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            import urllib.request
            ready = False
            for _ in range(120):
                try:
                    if urllib.request.urlopen("http://localhost:8000/health", timeout=2).status == 200:
                        ready = True; break
                except Exception:
                    time.sleep(2)
            if not ready:
                print("  [skip] 120s 未就绪"); continue
            # 抓 metrics（显存维）
            try:
                m = urllib.request.urlopen("http://localhost:8000/metrics", timeout=10).read().decode()
                kv = parse_metrics(m).get("kv_cache_usage_perc")
            except Exception:
                kv = None
            # 压测（吞吐/TTFT 维）
            bench = build_bench_cmd("http://localhost:8000", str(path), num_prompts=200)
            br = subprocess.run(bench, shell=True, capture_output=True, text=True, timeout=600)
            results[name] = {"bench_tail": br.stdout[-400:], "kv_cache_usage_perc": kv}
            print("  KV-Cache 利用率:", kv)
        finally:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    json.dump(results, open(OUT_ROOT / "s3_bench_compare.json", "w"), indent=2, ensure_ascii=False)
    print("\n[L3] 四向压测结果存 out/s3_bench_compare.json（吞吐/TTFT/显存对比）。")

def build_serve_cmd_str(path, tp=1):
    # 复用 s2 的 build_serve_cmd（若本 notebook 未定义则内联最小版）
    try:
        return build_serve_cmd(str(path), tp=tp, max_model_len=8192)
    except NameError:
        return "vllm serve %s --tensor-parallel-size %d --max-model-len 8192" % (path, tp)

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3_benchmark()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（reviewer 执行验证跳过真压测；真人跑时不设 SKIP_L3，L3 实证）。")


## 产物检查：吞吐/TTFT/显存四向对比

L3 跑完会在 `out/s3_bench_compare.json` 产四向（FP16/FP8/AWQ/SmoothQuant）压测结果。典型结论（H200×1，7B）：

| 方法 | 权重显存 | 吞吐（vs FP16）| TTFT | 精度（参考）|
|---|---|---|---|---|
| FP16 基线 | ~14GB | 1.0× | 基线 | 最高 |
| FP8 | ~7.5GB | ~1.5-2× 提速 | 略降 | ≈ FP16（H200 首选）|
| AWQ W4A16 | ~5GB | 访存墙场景提速 / 算力场景略拖 | 略升 | 略掉 |
| SmoothQuant W8A8 | ~7.5GB | ≈ INT8 | 略降 | 略掉 |

**关键解读**：
- **FP8 几乎全面最优**：H200 原生 FP8 Tensor Core，显存省一半、吞吐提升、精度≈FP16——这就是 OUTLINE 1.6「Hopper 上 FP8-Dynamic 全面替代 INT8-W8A8」的实证。
- **AWQ 省显存最狠但算力场景可能拖慢**：W4A16 每次要 dequant（FP16 GEMM 前解压 INT4），batch 大时 dequant 开销可能抵消访存红利——这就是 s2「TP 不是越大越好」之外的另一个"方法不是越激进越好"。
- **必须有 FP16 基线**：没有这一列，上面所有数字都是悬空的。

下一步 s4 讲这些压测/部署过程常见的报错诊断。
